The code presented here is (mostly) taken from `gtep/driver_t2k.py`

Global comment: we may want to turn off warnings for this notebook. That or try to resolve some of these warnings...

In [16]:
from gtep.gtep_data import ExpansionPlanningData
from gtep.gtep_model import ExpansionPlanningModel

#### Reading in data

In [17]:
# need some markdown or comments here to explain what these arguments are,
# how they map onto the different time horizons discussed in slide deck
data_object = ExpansionPlanningData(
    stages=3,
    num_reps=4,
    len_reps=24,
    num_commit=24,
    num_dispatch=1,
)

In [18]:
from pathlib import Path

data_path = (Path() / ".." / "data" / "5bus").resolve()
data_object.load_prescient(data_path)

# data_object.import_load_scaling  # do we want to do this?
# data_object.texas_case_study_updates  # do we want to do this?

c:\Users\agmoore\AppData\Local\anaconda3\envs\gtep\Lib\site-packages\egret\parsers\rts_gmlc\parser.py:254: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(file_name,
c:\Users\agmoore\AppData\Local\anaconda3\envs\gtep\Lib\site-packages\egret\parsers\rts_gmlc\parser.py:254: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(file_name,
c:\Users\agmoore\AppData\Local\anaconda3\envs\gtep\Lib\site-packages\egret\parsers\rts_gmlc\parser.py:254: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(file_name,
c:\Users\agmoore\AppData\Local\anaconda3\envs\gtep\Lib\site-packages\egret\parsers\rts_gmlc\parser.p

It would be good to have some code and/or markdown to explain what `ExpansionPlanningData` does; what options/capabilities it has; how to access the data within; etc.

In [19]:
data_object.representative_data
# describe what "representative data" means; why we have 4 elements here, etc.

In [20]:
elements = data_object.representative_data[0].data["elements"]
elements.keys()

dict_keys(['bus', 'load', 'shunt', 'area', 'branch', 'generator', 'storage'])

In [21]:
for gen, gen_data in elements["generator"].items():
    print(gen)
    print(gen_data)

3_CT
{'bus': 'bus3', 'in_service': True, 'mbase': 100.0, 'pg': 0.0, 'qg': 0.0, 'p_min': 8.0, 'p_max': 20.0, 'q_min': -20.0, 'q_max': 20.0, 'ramp_q': 3.0, 'fuel': 'G', 'unit_type': 'CT', 'area': '2', 'zone': '1', 'generator_type': 'thermal', 'p_fuel': {'data_type': 'fuel_curve', 'values': [(8.0, nan), (12.0, nan), (16.0, nan), (20.0, nan)]}, 'startup_fuel': [(1.0, 51.75)], 'non_fuel_startup_cost': 0.0, 'shutdown_cost': 0.0, 'agc_capable': True, 'p_min_agc': 8.0, 'p_max_agc': 20.0, 'ramp_agc': 3.0, 'ramp_up_60min': 180.0, 'ramp_down_60min': 180.0, 'fuel_cost': 0.75, 'startup_capacity': 8.0, 'shutdown_capacity': 8.0, 'min_up_time': 1.0, 'min_down_time': 1.0, 'initial_status': 24.0, 'initial_p_output': 14.0, 'initial_q_output': 0.0, 'lifetime': 3, 'spinning_reserve_frac': 0.1, 'quickstart_reserve_frac': 0.1, 'capital_multiplier': 1, 'extension_multiplier': 0, 'max_operating_reserve': 1, 'max_spinning_reserve': 1, 'max_quickstart_reserve': 1, 'ramp_up_rate': 0.1, 'ramp_down_rate': 0.1, 'emi

#### Creating the model

Need some markdown here: explain what config options can be set here. 

For general code usability it would be good to have a documented list of all the config options; for the sake of this notebook, maybe just a few examples is enough

In [22]:
mod_object = ExpansionPlanningModel(data=data_object)
# possible bug: if you pass config into the above call, the options aren't set. have
# to set them manually after creating the model

# print default config options (for now, let's just keep the default)
for a, b in mod_object.config.items():
    print(a, b)

include_investment True
include_commitment True
include_redispatch True
flow_model DC
time_period_subsets <abc.UninitializedConfigList object at 0x00000164DD93E9C0>
time_period_dict <pyomo.common.config.ConfigDict object at 0x00000164DD37BF40>
dispatch_randomization True
scale_loads True
scale_texas_loads False
thermal_generation False
renewable_generation False
storage False
transmission False
transmission_switching False
advanced_hydro False


In [23]:
mod_object.create_model()
mod_object.model

Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2025. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


[    0.00] Creating GTEP Model
investmentStage[1].year = 2025


Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2030. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


investmentStage[2].year = 2030


Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2035. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


investmentStage[3].year = 2035


In [30]:
mod_object.model.data